# iiq2img — Usage Examples

Fast IIQ-to-image converter for Phase One iXM-GS120 (120MP) raw files with georeferencing support.

In [1]:
from pathlib import Path

from iiq2img import (
    convert_iiq,
    batch_convert,
    extract_metadata,
    extract_geo_info,
)

In [2]:
input_dir = Path("PhaseOneSample")
output_dir = Path.cwd() / "output"

In [3]:
input_files = list(input_dir.glob("*.IIQ"))
first_file = input_files[0]

## Single File Conversion

In [10]:
# Pipeline.LIBRAW  — LibRaw PPG, accurate, ~3s (default)
# Pipeline.FAST    — cv2 EA + numba, ~5× faster, visually equivalent

result = convert_iiq(
    first_file,
    output_dir / (first_file.stem + ".jpg"),
    pipeline="fast",
)

print(f"Output:     {result.output_path}")
print("Pipeline:   Pipeline.FAST")
print(f"Resolution: {result.width}x{result.height}")
print(f"Time:       {result.elapsed_ms:.0f}ms")
print(f"Size:       {result.file_size_bytes / 1024 / 1024:.1f} MB")

Output:     /Users/nick/Documents/Projects/iiq2img/output/P0286625 copy 4.jpg
Pipeline:   Pipeline.FAST
Resolution: 12768x9564
Time:       1056ms
Size:       50.2 MB


## Bulk Conversion

In [5]:
results = batch_convert(
    input_dir=input_dir,
    output_dir=output_dir,
    output_format="jpg",
    compress_quality=90,
    workers=8,
    pipeline="fast",
)

  0%|          | 0/15 [00:00<?, ?img/s]

Done: 15 images in 13.8s (avg 918ms/image, 1.1 images/sec, 8 workers)


## Georeferencing

Adds spatial reference using GPS/IMU data from IIQ metadata.
- **JPEG/PNG**: sidecar world file (.jgw/.pgw)
- **TIFF**: embedded GeoTIFF with CRS and affine transform

In [6]:
# JPEG with world file
result = convert_iiq(first_file, output_dir / (first_file.stem + ".jpg"), georef=True)

In [7]:
# GeoTIFF with embedded CRS
result = convert_iiq(first_file, output_dir / (first_file.stem + ".tif"), georef=True)

In [8]:
# GSD and footprint from metadata
meta = extract_metadata("PhaseOneSample/P0286625.IIQ")
geo = extract_geo_info(
    meta["xmp"], focal_length_mm=80.0, image_width=12768, image_height=9564
)

print(f"Position:  {geo.latitude:.6f}, {geo.longitude:.6f}")
print(f"Altitude:  {geo.altitude_agl:.1f}m AGL")
print(f"GSD:       {geo.gsd * 100:.2f} cm/pixel")
print(f"Footprint: {geo.footprint_width:.1f}m x {geo.footprint_height:.1f}m")

Position:  -34.560007, 118.378565
Altitude:  25.8m AGL
GSD:       0.11 cm/pixel
Footprint: 14.2m x 10.6m
